In [ ]:
import os
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np

# Paths
csv_path = "Data_Entry_2017_v2020.csv"
img_dir = "images"

# Load CSV
df = pd.read_csv(csv_path)


df['Finding Labels'] = df['Finding Labels'].str.split('|')
desiredClasses = ['No Finding', 'Pneumothorax', 'Cardiomegaly', 'Mass', 'Nodule']
desiredClasses = ['Pneumothorax', 'Cardiomegaly', 'Mass', 'Nodule']

filtered_df = df[df['Finding Labels'].apply(lambda x: sum(label in desiredClasses for label in x) == 1)]
filtered_df = filtered_df.drop_duplicates(subset='Patient ID')
filtered_df

#Count classes
min = 32000
for i in desiredClasses:
    totalForClass = filtered_df['Finding Labels'].apply(lambda item: i in item).sum()
    if min > totalForClass:
        min = totalForClass
    print(f"{i}: {totalForClass}")




Pneumothorax: 893
Cardiomegaly: 1271
Mass: 1717
Nodule: 2520


np.int64(893)

In [ ]:
import tarfile
from pathlib import Path

tar_files = [
    'tars/images_001.tar.gz',
    'tars/images_002.tar.gz',
    'tars/images_003.tar.gz',
    'tars/images_004.tar.gz',
    'tars/images_005.tar.gz',
    'tars/images_006.tar.gz',
    'tars/images_007.tar.gz',
]  

# Setup output directory
output_dir = Path('./extracted_images') # final output will be /extracted_images/images
output_dir.mkdir(parents=True, exist_ok=True)

image_index_set = set(filtered_df['Image Index'])
filtered_df['Recovered_Image'] = 0

# Get only desired files directly from tar.gz so we dont blow our storage
for tar_path in tar_files:
    with tarfile.open(tar_path, 'r:gz') as tar:
        haha = tar.getmembers()
        for member in tar.getmembers():
            filename = Path(member.name).name
            filePath = output_dir / 'images' / filename
            if filePath.exists():
                filtered_df.loc[filtered_df['Image Index'] == filename, 'Recovered_Image'] = 1 #mark that image was found
                continue
            if filename in image_index_set:
                tar.extract(member, path=output_dir)
                filtered_df.loc[filtered_df['Image Index'] == filename, 'Recovered_Image'] = 1 #mark that image was found



#Drop all df entries that are not in the extracted_folder
filtered_df.drop(filtered_df[filtered_df['Recovered_Image'] == 0].index, inplace=True)
filtered_df.drop(columns=['Recovered_Image'], inplace=True)
filtered_df

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Sex,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
0,00000001_000.png,[Cardiomegaly],0,1,57,M,PA,2682,2749,0.143,0.143
3,00000002_000.png,[No Finding],0,2,80,M,PA,2500,2048,0.171,0.171
13,00000005_000.png,[No Finding],0,5,69,F,PA,2048,2500,0.168,0.168
21,00000006_000.png,[No Finding],0,6,81,M,PA,2500,2048,0.168,0.168
22,00000007_000.png,[No Finding],0,7,82,M,PA,2500,2048,0.168,0.168
...,...,...,...,...,...,...,...,...,...,...,...
54968,00013770_000.png,[No Finding],0,13770,32,M,PA,2500,2048,0.168,0.168
54969,00013771_000.png,[No Finding],0,13771,66,M,PA,2992,2991,0.143,0.143
54970,00013772_000.png,[No Finding],0,13772,32,M,PA,2678,2754,0.143,0.143
54971,00013773_000.png,[No Finding],0,13773,29,F,PA,2674,2991,0.143,0.143


In [61]:
filtered_df

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Sex,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
0,00000001_000.png,[Cardiomegaly],0,1,57,M,PA,2682,2749,0.143,0.143
3,00000002_000.png,[No Finding],0,2,80,M,PA,2500,2048,0.171,0.171
13,00000005_000.png,[No Finding],0,5,69,F,PA,2048,2500,0.168,0.168
21,00000006_000.png,[No Finding],0,6,81,M,PA,2500,2048,0.168,0.168
22,00000007_000.png,[No Finding],0,7,82,M,PA,2500,2048,0.168,0.168
...,...,...,...,...,...,...,...,...,...,...,...
54968,00013770_000.png,[No Finding],0,13770,32,M,PA,2500,2048,0.168,0.168
54969,00013771_000.png,[No Finding],0,13771,66,M,PA,2992,2991,0.143,0.143
54970,00013772_000.png,[No Finding],0,13772,32,M,PA,2678,2754,0.143,0.143
54971,00013773_000.png,[No Finding],0,13773,29,F,PA,2674,2991,0.143,0.143
